In [ ]:
# ============================================================
# TOPIC MODELING USING LATENT DIRICHLET ALLOCATION (LDA)
# ============================================================

# ------------------------------------------------------------
# 1. Import Required Libraries
# ------------------------------------------------------------

import pandas as pd
import re

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


# ------------------------------------------------------------
# 2. Create Sample Dataset
# ------------------------------------------------------------

documents = [
    "Artificial intelligence and machine learning are changing technology.",
    "Machine learning algorithms are used for artificial intelligence applications.",
    "Deep learning and neural networks are important parts of AI.",

    "Football is a popular sport played around the world.",
    "Cricket players practice batting bowling and fielding every day.",
    "Football teams train hard before important matches.",

    "Python Java and C++ are popular programming languages.",
    "Software developers use Python and Java to build applications.",
    "Programming and software development require logical thinking."
]

df = pd.DataFrame({
    "Document": documents
})

print("Original Documents:")
display(df)


# ------------------------------------------------------------
# 3. Text Preprocessing
# ------------------------------------------------------------

def preprocess_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df["Cleaned_Text"] = df["Document"].apply(preprocess_text)

print("After Preprocessing:")
display(df)


# ------------------------------------------------------------
# 4. Convert Text into Document-Term Matrix
# ------------------------------------------------------------

vectorizer = CountVectorizer(
    stop_words="english"
)

X = vectorizer.fit_transform(df["Cleaned_Text"])

print("Document-Term Matrix Shape:")
print(X.shape)


# ------------------------------------------------------------
# 5. Display Vocabulary
# ------------------------------------------------------------

words = vectorizer.get_feature_names_out()

print("\nVocabulary:")
print(words)


# ------------------------------------------------------------
# 6. Create LDA Model
# ------------------------------------------------------------

lda_model = LatentDirichletAllocation(
    n_components=3,
    random_state=42
)

lda_model.fit(X)


# ------------------------------------------------------------
# 7. Display Topics
# ------------------------------------------------------------

feature_names = vectorizer.get_feature_names_out()

print("\nTopics Found by LDA:\n")

for topic_index, topic in enumerate(lda_model.components_):

    print(f"Topic {topic_index + 1}:")

    top_words = topic.argsort()[-5:][::-1]

    for word_index in top_words:
        print(feature_names[word_index], end=" ")

    print("\n")


# ------------------------------------------------------------
# 8. Assign Topic to Each Document
# ------------------------------------------------------------

document_topics = lda_model.transform(X)

df["Topic"] = document_topics.argmax(axis=1) + 1

print("Documents with Assigned Topics:")
display(df[["Document", "Topic"]])


# ------------------------------------------------------------
# 9. Display Topic Probability for Each Document
# ------------------------------------------------------------

topic_probability = lda_model.transform(X)

topic_columns = [
    f"Topic_{i+1}_Probability"
    for i in range(3)
]

topic_df = pd.DataFrame(
    topic_probability,
    columns=topic_columns
)

final_df = pd.concat(
    [df[["Document", "Topic"]], topic_df],
    axis=1
)

print("Topic Probabilities:")
display(final_df)


# ------------------------------------------------------------
# 10. Display Top Words for Each Topic
# ------------------------------------------------------------

print("\nTop Words in Each Topic:\n")

for topic_index, topic in enumerate(lda_model.components_):

    top_words = topic.argsort()[-7:][::-1]

    words_list = [
        feature_names[i]
        for i in top_words
    ]

    print(
        f"Topic {topic_index + 1}: "
        + ", ".join(words_list)
    )


# ------------------------------------------------------------
# 11. Predict Topic for New Document
# ------------------------------------------------------------

new_document = [
    "Machine learning and artificial intelligence are useful technologies."
]

new_document_vector = vectorizer.transform(new_document)

topic_distribution = lda_model.transform(
    new_document_vector
)

predicted_topic = topic_distribution.argmax() + 1

print("\nNew Document:")
print(new_document[0])

print("\nTopic Probabilities:")
print(topic_distribution)

print("\nPredicted Topic:", predicted_topic)


# ------------------------------------------------------------
# 12. Topic Summary
# ------------------------------------------------------------

print("\nTopic Modeling Summary:")

for topic_index, topic in enumerate(lda_model.components_):

    top_words = topic.argsort()[-5:][::-1]

    words_list = [
        feature_names[i]
        for i in top_words
    ]

    print(
        f"Topic {topic_index + 1}: "
        + ", ".join(words_list)
    )

Original Documents:


,Document
0,Artificial intelligence and machine learning a...
1,Machine learning algorithms are used for artif...
2,Deep learning and neural networks are importan...
3,Football is a popular sport played around the ...
4,Cricket players practice batting bowling and f...
5,Football teams train hard before important mat...
6,Python Java and C++ are popular programming la...
7,Software developers use Python and Java to bui...
8,Programming and software development require l...


After Preprocessing:


,Document,Cleaned_Text
0,Artificial intelligence and machine learning a...,artificial intelligence and machine learning a...
1,Machine learning algorithms are used for artif...,machine learning algorithms are used for artif...
2,Deep learning and neural networks are importan...,deep learning and neural networks are importan...
3,Football is a popular sport played around the ...,football is a popular sport played around the ...
4,Cricket players practice batting bowling and f...,cricket players practice batting bowling and f...
5,Football teams train hard before important mat...,football teams train hard before important mat...
6,Python Java and C++ are popular programming la...,python java and c are popular programming lang...
7,Software developers use Python and Java to bui...,software developers use python and java to bui...
8,Programming and software development require l...,programming and software development require l...


Document-Term Matrix Shape:
(9, 43)

Vocabulary:
['ai' 'algorithms' 'applications' 'artificial' 'batting' 'bowling' 'build'
 'changing' 'cricket' 'day' 'deep' 'developers' 'development' 'fielding'
 'football' 'hard' 'important' 'intelligence' 'java' 'languages'
 'learning' 'logical' 'machine' 'matches' 'networks' 'neural' 'parts'
 'played' 'players' 'popular' 'practice' 'programming' 'python' 'require'
 'software' 'sport' 'teams' 'technology' 'thinking' 'train' 'use' 'used'
 'world']

Topics Found by LDA:

Topic 1:
learning machine intelligence artificial applications 

Topic 2:
programming important software popular java 

Topic 3:
football practice batting bowling fielding 

Documents with Assigned Topics:


,Document,Topic
0,Artificial intelligence and machine learning a...,1
1,Machine learning algorithms are used for artif...,1
2,Deep learning and neural networks are importan...,2
3,Football is a popular sport played around the ...,3
4,Cricket players practice batting bowling and f...,3
5,Football teams train hard before important mat...,3
6,Python Java and C++ are popular programming la...,2
7,Software developers use Python and Java to bui...,3
8,Programming and software development require l...,2


Topic Probabilities:


,Document,Topic,Topic_1_Probability,Topic_2_Probability,Topic_3_Probability
0,Artificial intelligence and machine learning a...,1,0.904017,0.048219,0.047764
1,Machine learning algorithms are used for artif...,1,0.915502,0.042143,0.042355
2,Deep learning and neural networks are importan...,2,0.044551,0.912876,0.042573
3,Football is a popular sport played around the ...,3,0.056058,0.058049,0.885893
4,Cricket players practice batting bowling and f...,3,0.042088,0.042019,0.915893
5,Football teams train hard before important mat...,3,0.048056,0.049431,0.902513
6,Python Java and C++ are popular programming la...,2,0.055965,0.883480,0.060555
7,Software developers use Python and Java to bui...,3,0.043394,0.045961,0.910645
8,Programming and software development require l...,2,0.047974,0.903253,0.048773



Top Words in Each Topic:

Topic 1: learning, machine, intelligence, artificial, applications, algorithms, used
Topic 2: programming, important, software, popular, java, python, networks
Topic 3: football, practice, batting, bowling, fielding, day, cricket

New Document:
Machine learning and artificial intelligence are useful technologies.

Topic Probabilities:
[[0.86544412 0.06773998 0.06681589]]

Predicted Topic: 1

Topic Modeling Summary:
Topic 1: learning, machine, intelligence, artificial, applications
Topic 2: programming, important, software, popular, java
Topic 3: football, practice, batting, bowling, fielding
